# Data Preprocessing

> 📘 **Python Mastery** · Module 13 — Machine Learning · Lesson 2/7

Real datasets arrive with holes, text where you need numbers, and columns measured
in wildly different units — this lesson turns that mess into model-ready matrices.

## 🎯 Learning Objectives

- Explain why preprocessing quality often decides model performance more than algorithm choice.
- Fill missing values with `SimpleImputer` using different strategies.
- Convert categorical text into numbers with `OrdinalEncoder` and `OneHotEncoder` — choosing correctly between them.
- Scale features with `StandardScaler` and `MinMaxScaler`, and explain when each fits.
- Spot and prevent **data leakage** by fitting transformers on training data only.
- Apply different transforms to different columns in one step with `ColumnTransformer`.
- Prove the payoff: KNN accuracy jumps once features are scaled.

## 1. Garbage In, Garbage Out

Models are amplifiers: feed them clean signal and they amplify insight; feed them
noise and they amplify noise. A five-minute look at raw rows usually reveals
everything that will hurt later — blanks, mixed types, impossible values.

**Syntax:**
```python
import pandas as pd

df = pd.read_csv("customers.csv")
df.isna().sum()          # how many holes per column?
df.describe()            # ranges: any absurd min/max?
df["city"].value_counts()  # what categories exist?
```

In [ ]:
# Craft a small, painfully realistic messy dataset (deterministic seed)
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
n = 12
df = pd.DataFrame({
    "city":        rng.choice(["Dhaka", "Chattogram", "Sylhet"], n),
    "age":         rng.integers(20, 60, n).astype(float),
    "income_taka": np.round(rng.normal(65000, 18000, n)),
    "purchased":   rng.integers(0, 2, n),          # our label
})
# Poke realistic holes into it
df.loc[2, "age"] = np.nan
df.loc[9, "age"] = np.nan
df.loc[5, "income_taka"] = np.nan
df.loc[7, "city"] = None

print(df)
print("\nMissing values per column:")
print(df.isna().sum())

In [ ]:
# The three health checks worth running on ANY new table
print(df.dtypes)
print("\nNumeric ranges:")
print(df[["age", "income_taka"]].describe().round(1))
print("\nCategory counts:")
print(df["city"].value_counts(dropna=False))

Three problems, three lessons ahead: **holes** (`NaN`), **text categories**
(`city`), and **incomparable scales** (`age` spans ~40 while `income` spans
~100,000).

## 2. Handling Missing Values

Deleting every row with a hole wastes data; models refuse `NaN` outright. The
middle path is **imputation**: replace each hole with a sensible stand-in — often
the column's mean (numeric) or the fixed string `"Unknown"` (categorical).

**Syntax:**
```python
from sklearn.impute import SimpleImputer

imp_mean     = SimpleImputer(strategy="mean")      # also: median, most_frequent
imp_constant = SimpleImputer(strategy="constant", fill_value="Unknown")

filled = imp_mean.fit_transform(numeric_df)        # returns a NumPy array
```

In [ ]:
# Mean-impute the numeric columns
import pandas as pd
from sklearn.impute import SimpleImputer

num_cols = ["age", "income_taka"]
print("Before:", df[num_cols].isna().sum().to_dict())

imp = SimpleImputer(strategy="mean")
filled_values = imp.fit_transform(df[num_cols])     # learns each column mean, fills holes
num_filled = pd.DataFrame(filled_values, columns=num_cols)

print("After :", num_filled.isna().sum().to_dict())
print("Means learned:", dict(zip(num_cols, imp.statistics_.round(1))))

In [ ]:
# Constant-impute the categorical column (keeps 'missingness' visible as its own category)
import pandas as pd
from sklearn.impute import SimpleImputer

imp_cat = SimpleImputer(strategy="constant", fill_value="Unknown")
city_filled = imp_cat.fit_transform(df[["city"]]).ravel()

print(pd.Series(city_filled).value_counts())
print("\nNote:", (city_filled == "Unknown").sum(), "row(s) now carry the Unknown category")

## 3. Encoding Categorical Data

Models compute arithmetic, not words — `"Dhaka"` must become numbers. The choice
matters:

| Encoder | Output | Use when |
|---|---|---|
| `OrdinalEncoder` | One integer column (S=0, M=1, …) | The categories have a real **order** (size, education level, rating) |
| `OneHotEncoder` | One 0/1 column per category | Categories are just **names** (city, colour, brand) |

Numbering unordered categories is dangerous: the model would invent fake maths like
`Sylhet (2) > Dhaka (1)`.

**Syntax:**
```python
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

OrdinalEncoder(categories=[["S", "M", "L"]])              # explicit, meaningful order
OneHotEncoder(sparse_output=False, handle_unknown="ignore")  # dense output, safe on new values
```

In [ ]:
# Ordinal encoding ONLY makes sense when order exists — give the order EXPLICITLY
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

sizes = pd.DataFrame({"tshirt": ["M", "S", "XL", "L", "M", "S", "L"]})

enc_alpha = OrdinalEncoder()                                        # alphabetical (bad!)
enc_ordered = OrdinalEncoder(categories=[["S", "M", "L", "XL"]])    # meaningful order

sizes["alpha_bad"]  = enc_alpha.fit_transform(sizes[["tshirt"]]).ravel().astype(int)
sizes["ordered_ok"] = enc_ordered.fit_transform(sizes[["tshirt"]]).ravel().astype(int)
print(sizes)
print("\nAlphabetical claims L < M < S < XL — nonsense for clothing.")

In [ ]:
# One-hot encode an UNORDERED category: city
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

clean = df.copy()
clean["city"] = clean["city"].fillna("Unknown")

ohe = OneHotEncoder(sparse_output=False)
city_dummies = ohe.fit_transform(clean[["city"]])

one_hot = pd.DataFrame(city_dummies.astype(int),
                       columns=ohe.get_feature_names_out(["city"]))
print(pd.concat([clean[["city"]].reset_index(drop=True), one_hot], axis=1))
print("\nEach row has exactly one 1 - no fake ordering introduced.")

In [ ]:
# Quick alternative: pd.get_dummies (fine for exploration, riskier for ML pipelines)
import pandas as pd

quick = pd.get_dummies(clean[["city"]], prefix="city").astype(int)
print(quick.head())

print("\nWhy pipelines prefer OneHotEncoder:")
print("- remembers its learned categories -> .transform() on future data matches columns")
print("- get_dummies run separately on train/test can produce DIFFERENT columns")

## 4. Feature Scaling

Distance-based models (KNN, SVM, K-Means) and gradient-based ones sum comparisons
across columns. If `income_taka` varies over tens of thousands while `age` varies
over tens, income shouts and age whispers — purely because of units, not importance.

| Scaler | Formula | Result | Prefer when |
|---|---|---|---|
| `StandardScaler` | `(x − mean) / std` | mean 0, std 1 | Data roughly bell-shaped; algorithms assuming centred input |
| `MinMaxScaler` | `(x − min) / (max − min)` | everything in [0, 1] | Bounded range needed; neural nets; known min/max |

**Syntax:**
```python
from sklearn.preprocessing import StandardScaler, MinMaxScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_num)   # learn stats, then apply
```

In [ ]:
# See the effect numerically: describe() before vs after
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler

num_cols = ["age", "income_taka"]
X_num = SimpleImputer(strategy="mean").fit_transform(clean[num_cols])
X_num = pd.DataFrame(X_num, columns=num_cols)

standard = pd.DataFrame(StandardScaler().fit_transform(X_num), columns=num_cols)
minmax   = pd.DataFrame(MinMaxScaler().fit_transform(X_num), columns=num_cols)

print("BEFORE (raw):");  print(X_num.describe().loc[["mean", "std", "min", "max"]].round(1))
print("\nStandardScaler ->"); print(standard.describe().loc[["mean", "std", "min", "max"]].round(2))
print("\nMinMaxScaler  ->"); print(minmax.describe().loc[["mean", "std", "min", "max"]].round(2))

Same information, new units: standardised columns hover around 0 with spread 1;
min-maxed columns live in [0, 1]. The *shape* never changes — only the ruler.

## 5. ⚠️ Data Leakage: Fit on TRAIN, Transform Both

Here is the trap almost every beginner falls into: calling `fit` on the **whole**
dataset. A fitted transformer stores statistics computed from whatever it saw —
if it saw the test rows, information from your "unseen" exam has leaked into
preprocessing, and every downstream score is quietly optimistic.

The iron pattern:

```python
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit AND transform on train only
X_test_scaled  = scaler.transform(X_test)        # transform only — NO second fit
```

In [ ]:
# WRONG vs RIGHT, side by side — watch the test-set statistics
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = clean[["age", "income_taka"]]
y = clean["purchased"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3,
                                                    random_state=42, stratify=y)
# ❌ WRONG: the scaler peeked at the test set
leaky_scaler = StandardScaler().fit(X)                 # fitted on ALL rows
test_leaky   = leaky_scaler.transform(X_test)

# ✅ RIGHT: the scaler learns from TRAIN rows alone
clean_scaler = StandardScaler().fit(X_train)           # train only
test_clean   = clean_scaler.transform(X_test)

print("Leaky  test-set means after scaling:",
      test_leaky.mean(axis=0).round(3))
print("Clean  test-set means after scaling:",
      test_clean.mean(axis=0).round(3))
print("(A clean transform leaves the test mean slightly OFF zero;")
print(" a leaky one lands suspiciously ON zero - test info crept in.)")

In [ ]:
# Same rule applies end-to-end: score both ways and compare
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(leaky_scaler.transform(X_train), y_train)
acc_leaky = accuracy_score(y_test, knn.predict(test_leaky))

knn.fit(clean_scaler.transform(X_train), y_train)
acc_clean = accuracy_score(y_test, knn.predict(test_clean))

print(f"accuracy with leaky scaling : {acc_leaky:.3f}")
print(f"accuracy with clean scaling : {acc_clean:.3f}")
print("\nWith plain scaling the gap here is small - the REAL damage comes from")
print("target-aware steps (feature selection, target encoding, imputation choices)")
print("where leakage can inflate scores massively. Build the CLEAN habit now.")

> 🔍 **Under the Hood:** a fitted scaler is nothing but frozen constants.
> `StandardScaler` stores `mean_` and `scale_`; `transform()` then does
> `(x − mean_) / scale_` — pure arithmetic, no further learning. That is exactly
> why the fit/transform split matters: `fit` is the only moment data can influence
> the constants. Once those constants are computed on train rows, transforming
> anything else cannot leak. The same logic extends to imputers (`statistics_`),
> encoders (`categories_`), and even deep-learning tokenisers.

## 6. ColumnTransformer: Different Transforms per Column

Numeric columns need imputing + scaling; categorical columns need imputing +
one-hotting. `ColumnTransformer` routes each column list to its own mini-pipeline
and glues the results side-by-side into one matrix.

**Syntax:**
```python
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline

preprocess = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(), StandardScaler()), ["age", "income"]),
    ("cat", make_pipeline(SimpleImputer(strategy="constant", fill_value="?"),
                          OneHotEncoder(handle_unknown="ignore")), ["city"]),
])
X_ready = preprocess.fit_transform(X_train)   # still fit on TRAIN only!
```

In [ ]:
# One object that does ALL preprocessing of our messy table
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

raw = df.copy()                       # back to the ORIGINAL messy frame (holes included)
features = raw.drop(columns="purchased")

preprocess = ColumnTransformer([
    ("nums", make_pipeline(SimpleImputer(strategy="mean"), StandardScaler()),
     ["age", "income_taka"]),
    ("cats", make_pipeline(SimpleImputer(strategy="constant", fill_value="Unknown"),
                           OneHotEncoder(handle_unknown="ignore")),
     ["city"]),
])

X_ready = preprocess.fit_transform(features)      # fit on ALL rows here for display only
names = preprocess.get_feature_names_out()
print(pd.DataFrame(X_ready, columns=names).round(2).head())
print("\nShape:", X_ready.shape, "- every value numeric, no NaN anywhere.")

This single `preprocess` object now handles future customers identically —
including cities it has never seen (`handle_unknown="ignore"` maps those to all
zeros instead of crashing). In lesson 7 we drop it straight into a `Pipeline`
so cross-validation stays leakage-free automatically.

## 7. Proof of Payoff: KNN Before vs After Scaling

Let's manufacture a case where units genuinely distort distances: whether a
customer buys depends mostly on **age**, but income (pure noise) ranges ~100× wider.
Unscaled, KNN picks neighbours by the loudest column — the noise.

In [ ]:
# Scaling rescue mission: same model, same split, only the ruler changes
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

rng = np.random.default_rng(42)
n = 300
age = rng.uniform(18, 70, n)
income = np.clip(rng.normal(60000, 25000, n), 8000, None)
y = (age < 42).astype(int)                      # truth: young folks buy
X = pd.DataFrame({"age": age, "income_taka": income})

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25,
                                          random_state=42, stratify=y)
knn = KNeighborsClassifier(n_neighbors=5)

# BEFORE: raw, unscaled
knn.fit(X_tr, y_tr)
before = accuracy_score(y_te, knn.predict(X_te))

# AFTER: scaled with train-fitted statistics
scaler = StandardScaler().fit(X_tr)
knn.fit(scaler.transform(X_tr), y_tr)
after = accuracy_score(y_te, knn.predict(scaler.transform(X_te)))

print(f"KNN accuracy WITHOUT scaling: {before:.2f}")
print(f"KNN accuracy WITH    scaling: {after:.2f}")
print(f"Free accuracy gained by fixing units: {after - before:+.2f}")

No new data, no new model, no tuning — just correct units, and accuracy leaps.
Whenever your model uses distances or gradients, scaling is not optional polish;
it is part of the model.

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Fitting scaler/imputer on all data before splitting | Test-set statistics leak into training; scores inflate | `fit` on train, `transform` on both |
| Ordinal-encoding unordered categories (cities) | Model invents fake order and fake distances | `OneHotEncoder` for names, `OrdinalEncoder` only for true ranks |
| Imputing before splitting | The imputed mean contains test-row information | Put the imputer inside a Pipeline / ColumnTransformer |
| Running `pd.get_dummies` separately on train and test | Columns can mismatch, crashing `.predict` | Use `OneHotEncoder` inside the pipeline |
| Dropping every row containing NaN | Small datasets shrink dangerously; bias creeps in | Impute sensibly; track missingness as a signal |
| Forgetting to save the fitted transformer | Tomorrow's inputs meet yesterday's wrong scale | Persist scaler together with the model (lesson 7) |

## 💡 Best Practices & Pro Tips

- **Wrap transforms in a `Pipeline`** so sklearn itself enforces fit-on-train during
  cross-validation — the machine makes leakage structurally impossible.
- **Inspect before you impute:** a whole column missing may mean a broken export,
  not a modelling problem.
- **Keep missingness as signal:** sometimes *"income was hidden"* predicts behaviour;
  a constant-fill category preserves that clue.
- **Scale tree-based models' inputs?** Usually unnecessary — trees split per-column
  and don't care about units; KNN/K-Means/SVM/neural nets absolutely do.
- **AI-engineering relevance:** production feature stores do exactly this work —
  encoders and scalers are saved artefacts served alongside the model, because
  serving-time inputs must be transformed with *training-time* constants.

## 📌 Summary

| Method | What it does | Example |
|---|---|---|
| `SimpleImputer(strategy="mean")` | Fills NaN with column statistic | `SimpleImputer(strategy="median").fit_transform(X)` |
| `SimpleImputer(strategy="constant", fill_value="Unknown")` | Fills NaN with a fixed value | Categorical holes |
| `OrdinalEncoder(categories=[[...]])` | Categories → integers honouring given order | T-shirt sizes |
| `OneHotEncoder(sparse_output=False, handle_unknown="ignore")` | Categories → 0/1 dummy columns | Cities, brands |
| `pd.get_dummies(df)` | Quick one-hot for exploration | Notebook prototyping |
| `StandardScaler` / `MinMaxScaler` | Rescale numeric columns | Before KNN, SVM, PCA |
| `ColumnTransformer([...])` | Route columns to different transforms | nums → scale, cats → one-hot |

Key takeaways:
- Preprocessing is where most real-world model gains hide — budget more time for
  it than for model selection.
- The iron rule: **fit on train, transform everywhere else** — for scalers,
  imputers, and encoders alike.
- Ordered categories deserve ordered codes; name-like categories deserve one-hot.
- Correctly scaled features gave KNN a massive free win in our demo.

## 🔗 Next Lesson

Continue to **[03_Train_Test_Split_CV](../03_Train_Test_Split_CV/notes.ipynb)** —
we keep saying "train only!" — time to master splitting and cross-validation properly.